In [ ]:
# Hotel Booking Cancellation Prediction
# Part 1 – Business Understanding (Chapter 2)
# Part 2 – Data Understanding (Chapter 3)
# 2.1 Collect initial data – import libraries and load the file
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREY = "#2b6cb0", "#dd6b20", "#a0aec0"
print("Libraries loaded")

In [ ]:
# Load the dataset by reading the CSV file directly (no upload button).
# Before running: drag hotel_bookings.csv into the Files panel on the left (folder icon).
# If the file is somewhere else (e.g. Google Drive), change FILE_PATH to that path.
FILE_PATH = "hotel_bookings.csv"

raw = pd.read_csv(FILE_PATH)
print("Loaded:", FILE_PATH, "| Rows:", raw.shape[0], "| Columns:", raw.shape[1])

In [ ]:
# 2.2 Describe data – size, columns and types
raw.head()

In [ ]:
raw.info()

In [ ]:
# Types of data (Chapter 3: nominal, ordinal, interval, ratio)
nominal = raw.select_dtypes(exclude="number").columns.tolist()
numeric = raw.select_dtypes(include="number").columns.tolist()
print(len(nominal), "text/nominal columns:", nominal)
print()
print(len(numeric), "numeric columns:", numeric)

In [ ]:
# 2.3 Explore data – descriptive statistics
raw.describe().T.round(2)

In [ ]:
# Categorical columns: number of different values and the most frequent value (mode)
raw.describe(include="object").T

In [ ]:
print(raw["is_canceled"].value_counts())
print((raw["is_canceled"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

raw["is_canceled"].value_counts().plot(kind="bar", color=[BLUE, ORANGE])
plt.xticks([0, 1], ["Not cancelled (0)", "Cancelled (1)"], rotation=0)
plt.title("Label: is_canceled"); plt.ylabel("Number of bookings"); plt.show()

In [ ]:
# 2.4 Verify data quality
# 1. Missing values per column
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"missing": missing, "percent": (missing / len(raw) * 100).round(2)})

In [ ]:
# 2-5. Noise, inconsistencies, redundancy, incorrect types
guests_raw = raw["adults"] + raw["children"].fillna(0) + raw["babies"]
nights_raw = raw["stays_in_weekend_nights"] + raw["stays_in_week_nights"]

quality = {
    "1. Missing cells (total)":               int(raw.isna().sum().sum()),
    "2. Noise: bookings with 0 guests":       int((guests_raw == 0).sum()),
    "2. Noise: bookings with 0 nights":       int((nights_raw == 0).sum()),
    "2. Noise: negative price (adr < 0)":     int((raw["adr"] < 0).sum()),
    "2. Noise: extreme price (adr > 1000)":   int((raw["adr"] > 1000).sum()),
    "3. Inconsistent: meal = 'Undefined'":    int((raw["meal"] == "Undefined").sum()),
    "3. Inconsistent: segment = 'Undefined'": int((raw["market_segment"] == "Undefined").sum()),
    "3. Inconsistent: channel = 'Undefined'": int((raw["distribution_channel"] == "Undefined").sum()),
    "4. Redundancy: duplicate rows":          int(raw.duplicated().sum()),
    "5. Incorrect type: month stored as text":  raw["arrival_date_month"].dtype == object or str(raw["arrival_date_month"].dtype) == "str",
    "5. Incorrect type: date stored as text":   raw["reservation_status_date"].dtype == object or str(raw["reservation_status_date"].dtype) == "str",
    "5. Incorrect type: children is float":    str(raw["children"].dtype),
}
pd.Series(quality, name="result").to_frame()

In [ ]:
# Boxplots to SEE outliers in key numeric columns
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["lead_time", "adr", "stays_in_week_nights"]):
    sns.boxplot(x=raw[col], ax=ax, color=BLUE)
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
# Part 3 – Data Preparation (Chapter 4)
df = raw.copy()
log = []          # Step 7: documentation log – every action is recorded here
def record(step, action, reason, rows_affected=""):
    log.append({"step": step, "action": action, "reason": reason, "rows/cols affected": rows_affected})

In [ ]:
# Step 1 – Select relevant attributes
# Variance check (like 'Remove Useless Attributes'): share of the most common value in each column
dominant = raw.apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0]).sort_values(ascending=False)
print("Share of the most frequent value (top 6):")
print((dominant.head(6) * 100).round(1).astype(str) + " %")

In [ ]:
df = df.drop(columns=["company"])
record(1, "Dropped column company", "94% missing values – almost empty", "1 column")
record(1, "Excluded reservation_status, reservation_status_date, assigned_room_type from modelling",
       "Data leakage – only known after the booking outcome", "3 columns")
print("Columns now:", df.shape[1])

In [ ]:
# Step 2 – Clean data
# 2a. Missing values – choosing the right imputation method (slides 36–43)
print("children mode:", df["children"].mode()[0], "| median:", df["children"].median())

df["children"] = df["children"].fillna(df["children"].mode()[0])
df["country"]  = df["country"].fillna("Unknown")
df["agent"]    = df["agent"].fillna(0)

record(2, "children missing -> mode (0)", "Only 4 rows; most frequent value", "4 rows")
record(2, "country missing -> 'Unknown'", "Keeps the bookings; country not known", "488 rows")
record(2, "agent missing -> 0 ('no agent')", "Missing means booking without an agent", "16,340 rows")
print("Missing values left:", df.isna().sum().sum())

In [ ]:
# 2b. Duplicates (redundancy)
before = len(df)
df = df.drop_duplicates()
record(2, "Removed exact duplicate rows", "Redundant records bias the model and the test", f"{before - len(df):,} rows")
print("Duplicates removed:", before - len(df))

In [ ]:
# 2c. Invalid / impossible values (noise)
before = len(df)
df = df[(df["adults"] + df["children"] + df["babies"]) > 0]
df = df[(df["stays_in_weekend_nights"] + df["stays_in_week_nights"]) > 0]
df = df[(df["adr"] >= 0) & (df["adr"] <= 1000)]
record(2, "Removed bookings with 0 guests, 0 nights, adr < 0 or adr > 1000", "Impossible values", f"{before - len(df):,} rows")
print("Invalid rows removed:", before - len(df))

In [ ]:
# 2d. Outliers – detect, then decide
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

pd.Series({c: iqr_outliers(df[c]) for c in ["lead_time", "adr", "stays_in_week_nights", "stays_in_weekend_nights"]},
          name="IQR outliers").to_frame()

In [ ]:
record(2, "Outliers detected with IQR rule but kept", "Long lead times / long stays are real behaviour and predictive", "0 rows")

In [ ]:
# Step 3 – Format and standardize
df["meal"] = df["meal"].replace("Undefined", "SC")
before = len(df)
df = df[(df["market_segment"] != "Undefined") & (df["distribution_channel"] != "Undefined")]
record(3, "meal 'Undefined' -> 'SC'", "Same meaning per dataset documentation", "")
record(3, "Removed 'Undefined' market_segment / distribution_channel", "No meaning", f"{before - len(df)} rows")

month_map = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6, "July": 7,
             "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}
df["arrival_month_num"] = df["arrival_date_month"].map(month_map)
df["arrival_date"] = pd.to_datetime(dict(year=df["arrival_date_year"], month=df["arrival_month_num"],
                                         day=df["arrival_date_day_of_month"]))
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])
df["children"] = df["children"].astype(int)
df["agent"] = df["agent"].astype(int)
record(3, "Month name -> number; dates -> datetime; children/agent -> int", "Correct types for calculations", "4 columns")

df = df.reset_index(drop=True)
df[["arrival_date_month", "arrival_month_num", "arrival_date", "children", "agent"]].dtypes

In [ ]:
# Step 4 – Construct new attributes
df["total_nights"]  = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"]  = df["adults"] + df["children"] + df["babies"]
df["has_kids"]      = ((df["children"] + df["babies"]) > 0).astype(int)
df["total_revenue"] = df["adr"] * df["total_nights"]
df["weekend_ratio"] = df["stays_in_weekend_nights"] / df["total_nights"]

prev_total = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
df["prev_cancel_rate"] = (df["previous_cancellations"] / prev_total.replace(0, np.nan)).fillna(0)

df["arrival_dow"] = df["arrival_date"].dt.dayofweek          # 0 = Monday
df["is_summer"]   = df["arrival_month_num"].isin([6, 7, 8]).astype(int)

lead_bins   = [-1, 7, 30, 90, 180, 365, 800]
lead_labels = ["0-7", "8-30", "31-90", "91-180", "181-365", "365+"]
df["lead_time_band"] = pd.cut(df["lead_time"], lead_bins, labels=lead_labels)

top_countries = df["country"].value_counts().head(15).index
df["country_grp"] = np.where(df["country"].isin(top_countries), df["country"], "Other")

df["room_mismatch"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)

record(4, "Created 11 new attributes (totals, ratios, binning, date features, grouping)", "Expose patterns to the model", "11 columns")
df[["total_nights", "total_guests", "total_revenue", "weekend_ratio", "prev_cancel_rate",
    "arrival_dow", "is_summer", "lead_time_band", "country_grp"]].head()

In [ ]:
# Step 5 – Integrate multiple datasets
record(5, "No integration needed", "All attributes are in a single file", "-")

In [ ]:
# Step 6 – Encode and normalize
pd.get_dummies(df[["deposit_type"]].head(6), dtype=int)

In [ ]:
record(6, "One-hot (linear models) / ordinal (trees) encoding + Z-normalization inside Pipelines",
       "Models need numeric input; fitted on training data only to avoid leakage", "")

In [ ]:
# Step 7 – Document the process and store the clean data
pd.DataFrame(log)

In [ ]:
before_after = pd.DataFrame({
    "Before cleaning": [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "After cleaning":  [df.shape[0],  df.shape[1],  int(df.isna().sum().sum()),  int(df.duplicated().sum())],
}, index=["Rows", "Columns", "Missing values", "Duplicate rows"])
before_after

In [ ]:
df.to_csv("hotel_bookings_clean.csv", index=False)
print("Clean data stored as hotel_bookings_clean.csv")

In [ ]:
# Exploratory Data Analysis (EDA) on the clean data
print("Overall cancellation rate after cleaning: %.1f %%" % (df["is_canceled"].mean() * 100))
(df.groupby("hotel")["is_canceled"].mean() * 100).round(1).to_frame("cancellation rate %")

In [ ]:
# Lead time (Figure 1 in the report)
lead = df.groupby("lead_time_band", observed=True)["is_canceled"].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lead.index.astype(str), lead, color=BLUE)
for i, v in enumerate(lead):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center")
ax.axhline(df["is_canceled"].mean() * 100, color=ORANGE, ls="--", label="overall rate")
ax.set_xlabel("Lead time (days between booking and arrival)"); ax.set_ylabel("Cancellation rate (%)")
ax.set_title("Cancellation rate rises sharply with lead time"); ax.legend()
plt.tight_layout(); plt.savefig("fig1_lead_time.png", dpi=150); plt.show()

print("Median lead time - cancelled:", df.loc[df.is_canceled == 1, "lead_time"].median(),
      "days | not cancelled:", df.loc[df.is_canceled == 0, "lead_time"].median(), "days")

In [ ]:
# Deposit type and special requests (Figure 2 in the report)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dep = df.groupby("deposit_type")["is_canceled"].mean().sort_values() * 100
axes[0].barh(dep.index, dep.values, color=[GREY, BLUE, ORANGE])
for i, v in enumerate(dep.values):
    axes[0].text(v + 1, i, f"{v:.0f}%", va="center")
axes[0].set_xlim(0, 110); axes[0].set_xlabel("Cancellation rate (%)"); axes[0].set_title("By deposit type")

req = df.groupby("total_of_special_requests")["is_canceled"].mean() * 100
axes[1].bar(req.index.astype(str), req.values, color=BLUE)
axes[1].set_xlabel("Number of special requests"); axes[1].set_ylabel("Cancellation rate (%)")
axes[1].set_title("By special requests")
plt.suptitle("Commitment signals: deposits and special requests")
plt.tight_layout(); plt.savefig("fig2_deposit_requests.png", dpi=150); plt.show()

In [ ]:
seg = (df.groupby("market_segment")["is_canceled"].mean() * 100).sort_values()
seg.plot(kind="barh", color=BLUE, figsize=(8, 4))
plt.xlabel("Cancellation rate (%)"); plt.title("Cancellation rate by market segment")
plt.tight_layout(); plt.show()

print("Parking requested :", round(df.loc[df.required_car_parking_spaces > 0, "is_canceled"].mean() * 100, 1), "%")
print("No parking        :", round(df.loc[df.required_car_parking_spaces == 0, "is_canceled"].mean() * 100, 1), "%")
print("Repeat guests     :", round(df.loc[df.is_repeated_guest == 1, "is_canceled"].mean() * 100, 1), "%")
print("New guests        :", round(df.loc[df.is_repeated_guest == 0, "is_canceled"].mean() * 100, 1), "%")

In [ ]:
month = df.groupby(["arrival_month_num", "hotel"])["is_canceled"].mean().unstack() * 100
month.plot(marker="o", figsize=(8, 4), color=[BLUE, ORANGE])
plt.xticks(range(1, 13)); plt.xlabel("Arrival month"); plt.ylabel("Cancellation rate (%)")
plt.title("Seasonality of cancellations by hotel"); plt.tight_layout(); plt.show()

In [ ]:
# Correlation analysis
corr = df.select_dtypes("number").drop(columns=["arrival_date_year"]).corr()["is_canceled"].drop("is_canceled").sort_values()
corr.plot(kind="barh", figsize=(7, 8), color=[ORANGE if v > 0 else BLUE for v in corr])
plt.title("Correlation with is_canceled"); plt.tight_layout(); plt.show()

In [ ]:
# Part 4 – Modeling (Chapter 5)
# 4.1 Select attributes and set the label
num_features = ["lead_time", "arrival_month_num", "arrival_date_week_number", "arrival_date_day_of_month",
                "arrival_dow", "stays_in_weekend_nights", "stays_in_week_nights", "total_nights",
                "adults", "children", "babies", "total_guests", "has_kids", "is_repeated_guest",
                "previous_cancellations", "previous_bookings_not_canceled", "prev_cancel_rate",
                "booking_changes", "days_in_waiting_list", "adr", "total_revenue", "weekend_ratio",
                "required_car_parking_spaces", "total_of_special_requests", "is_summer"]
cat_features = ["hotel", "meal", "country_grp", "market_segment", "distribution_channel",
                "reserved_room_type", "deposit_type", "customer_type"]

X = df[num_features + cat_features]     # attributes
y = df["is_canceled"]                    # label
print("Attributes:", X.shape[1], "(", len(num_features), "numeric +", len(cat_features), "nominal ) | Label: is_canceled")

In [ ]:
# 4.2 Generate test design – split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)

print("Train:", len(X_tr), "| Validation:", len(X_val), "| Test (hold-out):", len(X_test))
print("Cancel rate - train %.3f  val %.3f  test %.3f" % (y_tr.mean(), y_val.mean(), y_test.mean()))

In [ ]:
# 4.3 Preprocessing pipelines (Step 6 of Data Preparation, applied safely)
# For Logistic Regression and Naive Bayes: Z-normalization + one-hot (dummy) encoding
preprocess_linear = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)])

# For tree models: no scaling needed, categories -> integer codes
preprocess_tree = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_features)])

def evaluate(model, X_, y_, threshold=0.5):
    """Performance (Binominal Classification): returns the metrics from Chapter 6."""
    proba = model.predict_proba(X_)[:, 1]
    pred  = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=[0, 1]).ravel()
    return {"accuracy":    accuracy_score(y_, pred),
            "precision":   precision_score(y_, pred, zero_division=0),
            "recall":      recall_score(y_, pred),
            "specificity": tn / (tn + fp),
            "f1":          f1_score(y_, pred),
            "AUC":         roc_auc_score(y_, proba)}

In [ ]:
# 4.4 Select modeling techniques and build models
models = {
    "Baseline (majority class)": Pipeline([("pre", preprocess_linear), ("model", DummyClassifier(strategy="most_frequent"))]),
    "Logistic Regression":       Pipeline([("pre", preprocess_linear), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "Naive Bayes":               Pipeline([("pre", preprocess_linear), ("model", GaussianNB())]),
    "Decision Tree":             Pipeline([("pre", preprocess_tree),   ("model", DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Random Forest":             Pipeline([("pre", preprocess_tree),   ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))]),
    "Gradient Boosted Trees":    Pipeline([("pre", preprocess_tree),   ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
}

val_results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    val_results[name] = evaluate(model, X_val, y_val)
    print("trained:", name)

pd.DataFrame(val_results).T.round(3).sort_values("f1", ascending=False)

In [ ]:
# Cross-validation check
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name in ["Logistic Regression", "Decision Tree", "Random Forest"]:
    s = cross_val_score(models[name], X_train, y_train, cv=cv5, scoring="f1", n_jobs=-1)
    print(f"{name:22s} F1 = {s.mean():.3f} ± {s.std():.3f}")

In [ ]:
# Decision Tree – an interpretable model for managers
dt = models["Decision Tree"]
feature_names = num_features + cat_features            # order used by preprocess_tree
plt.figure(figsize=(22, 9))
plot_tree(dt.named_steps["model"], max_depth=3, feature_names=feature_names,
          class_names=["Not cancelled", "Cancelled"], filled=True, rounded=True, fontsize=9, impurity=False, proportion=True)
plt.title("Decision Tree – top 3 levels"); plt.show()

In [ ]:
# 4.5 Optimize model parameters (hyperparameter tuning)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

dt_search = GridSearchCV(models["Decision Tree"],
                         {"model__max_depth": [6, 10, 14, 20], "model__min_samples_leaf": [1, 10, 50]},
                         scoring="f1", cv=cv, n_jobs=-1)
dt_search.fit(X_train, y_train)
print("Decision Tree best:", dt_search.best_params_, "| CV F1 = %.3f" % dt_search.best_score_)

In [ ]:
rf_params = {"model__n_estimators":     [200, 300],
             "model__max_depth":        [None, 20, 30],
             "model__min_samples_leaf": [1, 2, 4],
             "model__max_features":     ["sqrt", 0.5]}
rf_search = RandomizedSearchCV(models["Random Forest"], rf_params, n_iter=5, scoring="f1",
                               cv=cv, n_jobs=1, random_state=RANDOM_STATE, verbose=1)
rf_search.fit(X_train, y_train)
print("Random Forest best:", rf_search.best_params_, "| CV F1 = %.3f" % rf_search.best_score_)

In [ ]:
gbt_params = {"model__learning_rate":     [0.05, 0.1, 0.2],
              "model__max_iter":          [200, 400, 600],
              "model__max_leaf_nodes":    [31, 63, 127],
              "model__min_samples_leaf":  [20, 50],
              "model__l2_regularization": [0.0, 1.0]}
gbt_search = RandomizedSearchCV(models["Gradient Boosted Trees"], gbt_params, n_iter=10, scoring="f1",
                                cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1)
gbt_search.fit(X_train, y_train)
print("Gradient Boosted Trees best:", gbt_search.best_params_, "| CV F1 = %.3f" % gbt_search.best_score_)

In [ ]:
# Part 5 – Evaluation (Chapter 6)
# 5.1 Performance on the hold-out test set (Table 5 in the report)
for name in ["Baseline (majority class)", "Logistic Regression", "Naive Bayes"]:
    models[name].fit(X_train, y_train)

final_models = {
    "Baseline (majority class)":      models["Baseline (majority class)"],
    "Logistic Regression":            models["Logistic Regression"],
    "Naive Bayes":                    models["Naive Bayes"],
    "Decision Tree (tuned)":          dt_search.best_estimator_,
    "Random Forest (tuned)":          rf_search.best_estimator_,
    "Gradient Boosted Trees (tuned)": gbt_search.best_estimator_,
}
test_results = pd.DataFrame({n: evaluate(m, X_test, y_test) for n, m in final_models.items()}).T
test_results.round(3).sort_values("f1", ascending=False)

In [ ]:
best_name  = test_results["f1"].idxmax()
best_model = final_models[best_name]
print("Final model (highest F1 on the test set):", best_name)
print()
print(classification_report(y_test, best_model.predict(X_test), target_names=["Not cancelled", "Cancelled"]))

In [ ]:
# 5.2 Confusion matrix (Figure 3 in the report)
cm = confusion_matrix(y_test, best_model.predict(X_test))
ConfusionMatrixDisplay(cm, display_labels=["Not cancelled", "Cancelled"]).plot(cmap="Blues", values_format=",")
plt.title(f"Confusion matrix – {best_name}")
plt.savefig("fig3_confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Caught {tp:,} of {tp + fn:,} cancellations (recall {tp / (tp + fn):.1%}) | missed {fn:,} | false alarms {fp:,}")

In [ ]:
# 5.3 ROC curve and AUC
plt.figure(figsize=(6, 5))
for name in ["Logistic Regression", "Naive Bayes", "Decision Tree (tuned)", best_name]:
    proba = final_models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess (AUC = 0.5)")
plt.xlabel("False positive rate (1 - specificity)"); plt.ylabel("True positive rate (recall)")
plt.title("ROC curves on the test set"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig("fig_roc.png", dpi=150); plt.show()

In [ ]:
# 5.4 Lift chart
proba_best = best_model.predict_proba(X_test)[:, 1]
lift = pd.DataFrame({"actual": y_test.values, "proba": proba_best}).sort_values("proba", ascending=False)
lift["decile"] = np.repeat(np.arange(1, 11), int(np.ceil(len(lift) / 10)))[:len(lift)]
dec = lift.groupby("decile")["actual"].agg(cancellations="sum", bookings="count")
dec["cancel_rate_%"]   = (dec["cancellations"] / dec["bookings"] * 100).round(1)
dec["cumulative_%_of_all_cancellations"] = (dec["cancellations"].cumsum() / dec["cancellations"].sum() * 100).round(1)
dec["lift"] = (dec["cancellations"] / dec["bookings"] / y_test.mean()).round(2)
display(dec)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dec.index, dec["cancellations"], color=BLUE)
ax2 = ax.twinx(); ax2.plot(dec.index, dec["cumulative_%_of_all_cancellations"], color=ORANGE, marker="o")
ax.set_xlabel("Decile (1 = highest predicted risk)"); ax.set_ylabel("Cancellations in decile")
ax2.set_ylabel("Cumulative % of all cancellations", color=ORANGE); ax2.set_ylim(0, 105)
ax.set_xticks(range(1, 11)); plt.title("Lift chart – cancellations are concentrated in the top deciles")
plt.tight_layout(); plt.show()

In [ ]:
# 5.5 Cost of errors – which mistake is more expensive?
COST_FN, COST_FP = 180, 20      # euros – business assumptions

def total_cost(model, threshold=0.5):
    pred = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return fn * COST_FN + fp * COST_FP, fn, fp

rows = {}
for name, m in final_models.items():
    cost, fn_, fp_ = total_cost(m)
    rows[name] = {"false negatives": fn_, "false positives": fp_, "total cost (€)": cost}
cost_table = pd.DataFrame(rows).T.sort_values("total cost (€)")
cost_table["saving vs baseline (€)"] = cost_table.loc["Baseline (majority class)", "total cost (€)"] - cost_table["total cost (€)"]
cost_table

In [ ]:
# 5.6 Changing the threshold
thr_rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    m = evaluate(best_model, X_test, y_test, threshold=t)
    cost, fn_, fp_ = total_cost(best_model, t)
    thr_rows.append({"threshold": t, "accuracy": m["accuracy"], "recall": m["recall"], "precision": m["precision"],
                     "false positives": fp_, "false negatives": fn_, "total cost (€)": cost})
thr = pd.DataFrame(thr_rows).set_index("threshold")
display(thr.round(3))
print("Lowest-cost threshold:", thr["total cost (€)"].idxmin())

In [ ]:
# 5.7 Which attributes drive the predictions?
sample = X_test.sample(5000, random_state=RANDOM_STATE)
perm = permutation_importance(best_model, sample, y_test.loc[sample.index],
                              n_repeats=5, scoring="f1", random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

importance.head(10)[::-1].plot(kind="barh", color=BLUE, figsize=(8, 5))
plt.xlabel("Drop in F1 when the attribute is shuffled"); plt.title("Top 10 most important attributes")
plt.tight_layout(); plt.savefig("fig3b_feature_importance.png", dpi=150); plt.show()
importance.head(10).round(4)

In [ ]:
# 5.8 Go / No-Go decision for deployment
b = test_results.loc[best_name]
checks = pd.DataFrame({
    "criterion": ["F1 ≥ 0.70", "AUC ≥ 0.85 (good ranking)", "Recall ≥ 0.75 (catches most cancellations)",
                  "Beats the baseline", "Cheaper than doing nothing", "Uses only booking-time data (no leakage)"],
    "result":    [f"{b['f1']:.3f}", f"{b['AUC']:.3f}", f"{b['recall']:.3f}",
                  f"F1 {b['f1']:.3f} vs 0.000", f"saves €{cost_table.loc[best_name, 'saving vs baseline (€)']:,.0f} on the test set", "yes"],
    "pass":      [b["f1"] >= 0.70, b["AUC"] >= 0.85, b["recall"] >= 0.75, True,
                  cost_table.loc[best_name, "saving vs baseline (€)"] > 0, True]})
display(checks)
print("DECISION:", "GO ✅ - deploy the model" if checks["pass"].all() else "NO-GO ⚠️ - review the failed criteria")

In [ ]:
# Part 6 – Deployment
from google.colab import files

joblib.dump({"model": best_model, "num": num_features, "cat": cat_features,
             "top_countries": list(top_countries)}, "model.joblib", compress=3)
print("Saved model.joblib")

files.download("model.joblib")
for f in ["fig1_lead_time.png", "fig2_deposit_requests.png", "fig3_confusion_matrix.png",
          "fig3b_feature_importance.png", "fig_roc.png"]:
    files.download(f)

In [ ]:
# Try it: predict one new booking
new_booking = {
    "hotel": "City Hotel", "lead_time": 200, "arrival_month_num": 8, "arrival_date_week_number": 33,
    "arrival_date_day_of_month": 15, "arrival_dow": 4, "stays_in_weekend_nights": 1, "stays_in_week_nights": 2,
    "adults": 2, "children": 0, "babies": 0, "meal": "BB", "country_grp": "PRT",
    "market_segment": "Online TA", "distribution_channel": "TA/TO", "is_repeated_guest": 0,
    "previous_cancellations": 0, "previous_bookings_not_canceled": 0, "reserved_room_type": "A",
    "booking_changes": 0, "deposit_type": "No Deposit", "days_in_waiting_list": 0,
    "customer_type": "Transient", "adr": 120.0, "required_car_parking_spaces": 0, "total_of_special_requests": 0,
}
b = new_booking
nights = b["stays_in_weekend_nights"] + b["stays_in_week_nights"]
b.update(total_nights=nights, total_guests=b["adults"] + b["children"] + b["babies"],
         has_kids=int(b["children"] + b["babies"] > 0), total_revenue=b["adr"] * nights,
         weekend_ratio=b["stays_in_weekend_nights"] / nights, is_summer=int(b["arrival_month_num"] in (6, 7, 8)),
         prev_cancel_rate=0.0)

p = best_model.predict_proba(pd.DataFrame([b])[num_features + cat_features])[0, 1]
level = "HIGH" if p >= 0.6 else "MEDIUM" if p >= 0.35 else "LOW"
print(f"Probability of cancellation: {p:.1%}  ->  {level} risk")
print(f"Revenue at risk: €{p * b['total_revenue']:,.0f}")